# Get metrics from fused run

In [2]:
from src.segmentation_pipeline import segmentation_pipeline
from src.model_run import start_prediction_machine, detectron2_instances
from src.classic_instances import instance_regions, make_instance_gif
from src.fusion import fusion_of_masks
from src.classifier import classify_instances, load_classifiers, Category_Classifier
from src.coco_json_functions import *
from src.model_run import CLASSES
from src.instance_based_evaluation import compare_coco
from tqdm import tqdm

from collections import defaultdict
import cv2
import numpy as np
from matplotlib import pyplot as plt


from sklearn.base import TransformerMixin, BaseEstimator
class SelectColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
    def fit(self, xs, ys, **params):
        return self
    def transform(self, xs):
        return xs[self.columns]
    
from pathlib import Path

import json
import cv2
import numpy as np
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from scipy.optimize import linear_sum_assignment
from detectron2 import model_zoo
from collections import defaultdict
from pycocotools import mask as maskUtils

In [ ]:
eval = "/home/mellanie/Desktop/container/code/detectron2/model1/test.json"

# Classifiers
bmp_cls = "/home/mellanie/Desktop/container/model/classifiers/bumps_embedded_classifier.pkl"
bg_cls = "/home/mellanie/Desktop/container/model/classifiers/bg_regular_classifier.pkl"


model_path = "/home/mellanie/Desktop/container/model/segmentation_models/model2_NMS_0.5.pth"
multi_class = False


# predictor is the segmentaion model as a variable
predictor = start_prediction_machine(path_to_model=model_path, multi_class=multi_class)

# Make the classifier class
bmp_predictor, bg_predictor = load_classifiers(bmp_cls, bg_cls)
classifier = Category_Classifier(None, None, None, None, bmp_predictor, bg_predictor)


/home/mellanie/miniforge3/envs/py3_image_analysis/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mellanie/miniforge3/envs/py3_image_analysis/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/mellanie/miniforge3/envs/py3_image_analysis/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Tr

In [ ]:

def instance_segmentation_pipeline(img_path):
    
    #1. Classic Segmentation
    root, hairs = segmentation_pipeline(img_path, 
                                        max_hole_size=205, 
                                        min_area=100, 
                                        max_distance=20, 
                                        circularity_threshold=0.8, 
                                        verbose=False, 
                                        show_overlay=False) 


    #2. Separate Classic Segments by Instance
    #   get a list of each unique instance mask of root hairs
    classic_masks = instance_regions(hairs, root)
    
    #3. Get the Model outputs
    #   Instances is a list of all found instances in the image, including bounding box, mask, label, and label id
    model_instances = detectron2_instances(predictor, img_path, verbose=True)

    # 4. Fusion
    fused_instances = fusion_of_masks(classic_masks, model_instances)

    #5. Classify
    classified_instances = classify_instances(fused_instances, root, img_path, classifier)


    return classified_instances


In [5]:
# ============================================================
# STEP 2 — Load ground truth from test.json
# ============================================================
with open(eval) as f:
    coco_gt = json.load(f)

IMAGE_ROOT = "/home/mellanie/Desktop/Annotated_dataset/"

gt_by_image = defaultdict(list)
for ann in coco_gt["annotations"]:
    gt_by_image[ann["image_id"]].append(ann["bbox"])

# image_id -> (file_name, ecotype)
image_meta = {
    img["id"]: (img["file_name"], img["ecotype"])
    for img in coco_gt["images"]
}

In [6]:
def polygon_to_mask(segmentation, height, width):
    rles = maskUtils.frPyObjects(segmentation, height, width)
    rle = maskUtils.merge(rles)
    return maskUtils.decode(rle).astype(bool)

In [7]:
def compute_mask_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0


In [8]:
def match_masks_with_pairs(gt_masks, pred_masks, iou_threshold):
    if len(gt_masks) == 0 or len(pred_masks) == 0:
        return [], list(range(len(pred_masks))), list(range(len(gt_masks)))

    iou_matrix = np.zeros((len(gt_masks), len(pred_masks)))
    for i, gt in enumerate(gt_masks):
        for j, pred in enumerate(pred_masks):
            iou_matrix[i, j] = compute_mask_iou(gt, pred)

    gt_idx, pred_idx = linear_sum_assignment(-iou_matrix)

    matched_pairs = []
    matched_gt, matched_pred = set(), set()
    for g, p in zip(gt_idx, pred_idx):
        if iou_matrix[g, p] >= iou_threshold:
            matched_pairs.append((g, p))
            matched_gt.add(g)
            matched_pred.add(p)

    unmatched_pred = [p for p in range(len(pred_masks)) if p not in matched_pred]
    unmatched_gt = [g for g in range(len(gt_masks)) if g not in matched_gt]
    return matched_pairs, unmatched_pred, unmatched_gt

In [9]:
IOU_THRESHOLD = 0.50

gt_segs_by_image = defaultdict(list) 
for ann in coco_gt["annotations"]:
    gt_segs_by_image[ann["image_id"]].append(ann["segmentation"])

instance_stats_by_ecotype = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
pixel_stats_by_ecotype = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

for image_id, gt_segs in gt_segs_by_image.items():
    file_name, ecotype = image_meta[image_id]

    im = cv2.imread(file_name)
    if im is None:
        print(f"Could not read: {file_name}")
        continue

    height, width = im.shape[:2]

    # Rasterize ground truth polygons into full-size boolean masks
    gt_masks = [polygon_to_mask(seg, height, width) for seg in gt_segs]

    # --- Run the classic pipeline instead of predictor(im) ---
    classified_instances = instance_segmentation_pipeline(file_name)
    pred_masks = [inst["mask"].astype(bool) for inst in classified_instances]

    matched_pairs, unmatched_pred, unmatched_gt = match_masks_with_pairs(gt_masks, pred_masks, IOU_THRESHOLD)

    # --- Instance-level counts ---
    tp = len(matched_pairs)
    fp = len(unmatched_pred)
    fn = len(unmatched_gt)
    instance_stats_by_ecotype[ecotype]["tp"] += tp
    instance_stats_by_ecotype[ecotype]["fp"] += fp
    instance_stats_by_ecotype[ecotype]["fn"] += fn

    # --- Pixel-level counts, only on matched instance pairs ---
    for gt_idx, pred_idx in matched_pairs:
        gt_mask = gt_masks[gt_idx]
        pred_mask = pred_masks[pred_idx]

        pixel_tp = np.logical_and(gt_mask, pred_mask).sum()
        pixel_fp = np.logical_and(~gt_mask, pred_mask).sum()
        pixel_fn = np.logical_and(gt_mask, ~pred_mask).sum()

        pixel_stats_by_ecotype[ecotype]["tp"] += pixel_tp
        pixel_stats_by_ecotype[ecotype]["fp"] += pixel_fp
        pixel_stats_by_ecotype[ecotype]["fn"] += pixel_fn

    print(f"{file_name}: TP={tp}, FP={fp}, FN={fn}")

/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000001.tif: TP=13, FP=18, FN=45
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000002.tif: TP=16, FP=17, FN=50
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000003.tif: TP=12, FP=21, FN=59
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000005.tif: TP=14, FP=7, FN=30
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000006.tif: TP=13, FP=13, FN=45
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000008.tif: TP=6, FP=3, FN=18
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000009.tif: TP=7, FP=7, FN=31
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000010.tif: TP=7, FP=9, FN=27
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000013.tif: TP=7, FP=8, FN=20
/home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/0

In [10]:
def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1


In [11]:
def print_report(stats_by_ecotype, title):
    print(f"\n{'='*70}\n{title}\n{'='*70}")
    print(f"{'Ecotype':<12}{'TP':>10}{'FP':>10}{'FN':>10}{'Precision':>12}{'Recall':>10}{'F1':>8}")
    print("-" * 75)

    total_tp = total_fp = total_fn = 0
    for ecotype, s in sorted(stats_by_ecotype.items()):
        precision, recall, f1 = precision_recall_f1(s["tp"], s["fp"], s["fn"])
        print(f"{ecotype:<12}{s['tp']:>10}{s['fp']:>10}{s['fn']:>10}{precision:>12.4f}{recall:>10.4f}{f1:>8.4f}")
        total_tp += s["tp"]
        total_fp += s["fp"]
        total_fn += s["fn"]

    micro_p, micro_r, micro_f1 = precision_recall_f1(total_tp, total_fp, total_fn)
    print("-" * 75)
    print(f"Overall: Precision={micro_p:.4f}, Recall={micro_r:.4f}, F1={micro_f1:.4f}")

# pooled TP/FP/FN across all ecotypes)
print_report(instance_stats_by_ecotype, "INSTANCE-BASED EVALUATION")
print_report(pixel_stats_by_ecotype, "PIXEL-BASED EVALUATION")


INSTANCE-BASED EVALUATION
Ecotype             TP        FP        FN   Precision    Recall      F1
---------------------------------------------------------------------------
Bay-0              135       170       455      0.4426    0.2288  0.3017
Col-0*              18        45       211      0.2857    0.0786  0.1233
Cvi-0              131       158       525      0.4533    0.1997  0.2772
Ge-0               151       217       556      0.4103    0.2136  0.2809
Hs-0                75       109       363      0.4076    0.1712  0.2412
Kondara            170       152       450      0.5280    0.2742  0.3609
Lip-0              108       155       459      0.4106    0.1905  0.2602
No-0               129       181       548      0.4161    0.1905  0.2614
Oy-0               111       122       353      0.4764    0.2392  0.3185
Pu2-23             131       158       491      0.4533    0.2106  0.2876
Shahdara            98        75       246      0.5665    0.2849  0.3791
St-0               13

In [ ]:
import pandas as pd

def build_results_csv(instance_stats_by_ecotype, pixel_stats_by_ecotype, output_path="results.csv"):
    rows = []

    for ecotype in sorted(instance_stats_by_ecotype.keys()):
        inst = instance_stats_by_ecotype[ecotype]
        pix = pixel_stats_by_ecotype[ecotype]

        inst_p, inst_r, inst_f1 = precision_recall_f1(inst["tp"], inst["fp"], inst["fn"])
        pix_p, pix_r, pix_f1 = precision_recall_f1(pix["tp"], pix["fp"], pix["fn"])

        rows.append({
            "genotype": ecotype,
            "instance_precision": inst_p,
            "instance_recall": inst_r,
            "instance_f1": inst_f1,
            "pixel_precision": pix_p,
            "pixel_recall": pix_r,
            "pixel_f1": pix_f1,
        })

    df = pd.DataFrame(rows)

    # --- Totals row (micro-average: pooled TP/FP/FN across all genotypes) ---
    total_inst_tp = sum(s["tp"] for s in instance_stats_by_ecotype.values())
    total_inst_fp = sum(s["fp"] for s in instance_stats_by_ecotype.values())
    total_inst_fn = sum(s["fn"] for s in instance_stats_by_ecotype.values())
    total_inst_p, total_inst_r, total_inst_f1 = precision_recall_f1(total_inst_tp, total_inst_fp, total_inst_fn)

    total_pix_tp = sum(s["tp"] for s in pixel_stats_by_ecotype.values())
    total_pix_fp = sum(s["fp"] for s in pixel_stats_by_ecotype.values())
    total_pix_fn = sum(s["fn"] for s in pixel_stats_by_ecotype.values())
    total_pix_p, total_pix_r, total_pix_f1 = precision_recall_f1(total_pix_tp, total_pix_fp, total_pix_fn)

    total_row = pd.DataFrame([{
        "genotype": "TOTAL (micro-avg)",
        "instance_precision": total_inst_p,
        "instance_recall": total_inst_r,
        "instance_f1": total_inst_f1,
        "pixel_precision": total_pix_p,
        "pixel_recall": total_pix_r,
        "pixel_f1": total_pix_f1,
    }])

    df = pd.concat([df, total_row], ignore_index=True)
    df.to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
    return df

build_results_csv(instance_stats_by_ecotype, pixel_stats_by_ecotype, "fused_pipeline_pixelIoU_results.csv")

Saved to classic_pipeline_pixelIoU_results.csv


,genotype,instance_precision,instance_recall,instance_f1,pixel_precision,pixel_recall,pixel_f1
0,Bay-0,0.442623,0.228814,0.301676,0.723535,0.800132,0.759908
1,Col-0*,0.285714,0.078603,0.123288,0.689544,0.759243,0.722717
2,Cvi-0,0.453287,0.199695,0.277249,0.715473,0.825525,0.766569
3,Ge-0,0.410326,0.213579,0.280930,0.799672,0.733001,0.764886
4,Hs-0,0.407609,0.171233,0.241158,0.659400,0.817632,0.730040
5,Kondara,0.527950,0.274194,0.360934,0.692790,0.816078,0.749397
6,Lip-0,0.410646,0.190476,0.260241,0.850102,0.688732,0.760956
7,No-0,0.416129,0.190547,0.261398,0.708080,0.809332,0.755328
8,Oy-0,0.476395,0.239224,0.318508,0.690967,0.796147,0.739837
9,Pu2-23,0.453287,0.210611,0.287596,0.759426,0.765079,0.762242
